# Reading a page and a count through the protected helpers

`_fetch` answers every row a query returned; `_fetchval` answers the first value of its first row. Together with `_execute` and `_fetchrow` they are the driver's four protected answers.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The two helpers come from `PostgresResource`; the example supplies only the connection that answers them and the domain method that calls them.

In [1]:
import asyncio
from typing import Any

from aoa.action_machine.exceptions import HandleError
from aoa.action_machine.resources.postgres import PostgresResource

## 2. Define the objects used in this case

The recording connection answers the two reads and records the statement with its positional parameters. The pool hands out one connection; neither class parses SQL or stores database rows.

In [2]:
class RecordingConnection:
    """A connection that answers the two new reads and records what it was asked for."""

    def __init__(self) -> None:
        self.rows: list[dict[str, Any]] = []
        self.value: Any = None
        self.calls: list[tuple[str, tuple[Any, ...]]] = []
        self.closed = False

    async def fetch(self, query: str, *params: Any) -> list[dict[str, Any]]:
        """Answer the seeded page and record the statement with its parameters."""
        self._check_open()
        self.calls.append((query, params))
        return [dict(row) for row in self.rows]

    async def fetchval(self, query: str, *params: Any) -> Any:
        """Answer the seeded value and record the statement with its parameters."""
        self._check_open()
        self.calls.append((query, params))
        return self.value

    async def close(self) -> None:
        """Mark the physical connection closed."""
        self.closed = True

    def _check_open(self) -> None:
        """Refuse a read once the connection is closed, as the driver does."""
        if self.closed:
            raise RuntimeError("connection is closed")


class RecordingPool:
    """A pool of one observable connection, without a server."""

    def __init__(self, size: int = 1) -> None:
        """Create the fixed capacity this case borrows from."""
        self.connections = [RecordingConnection() for _ in range(size)]
        self.free = list(self.connections)
        self.held: set[RecordingConnection] = set()
        self.closed = False

    async def acquire(self, *, timeout: float | None = None) -> RecordingConnection:
        """Hand out a free connection."""
        connection = self.free.pop()
        self.held.add(connection)
        return connection

    async def release(self, connection: RecordingConnection) -> None:
        """Take one borrowed connection back."""
        self.held.remove(connection)
        self.free.append(connection)

    async def close(self) -> None:
        """Close the pool after every borrower has returned."""
        assert not self.held
        for connection in self.connections:
            await connection.close()
        self.closed = True

## 3. The domain resource

A semantic method states its own ordering and bound and reads the page through `_fetch`; the counter reads one value through `_fetchval` and decides what `None` means.

In [3]:
class NotesResource(PostgresResource):
    """The notes domain's resource: a page through ``_fetch``, a count through ``_fetchval``."""

    async def list_notes(self, limit: int = 2, offset: int = 0) -> list[dict[str, Any]]:
        """Return one ordered, bounded page of notes as plain mappings."""
        rows = await self._fetch(
            "SELECT id, text FROM notes ORDER BY id LIMIT $1 OFFSET $2",
            (limit, offset),
        )
        return [dict(row) for row in rows]

    async def count_notes(self) -> int:
        """Return how many notes exist, with the helper's ``None`` answered as zero."""
        total = await self._fetchval("SELECT count(*) FROM notes")
        return 0 if total is None else int(total)

## 4. A page with rows

The helper answers every row in the driver's order, and the statement's parameters reach the connection positionally.

In [4]:
pool = RecordingPool()
resource = NotesResource(pool=pool)
await resource.open()
connection = pool.connections[0]

connection.rows = [{"id": "n1", "text": "first"}, {"id": "n2", "text": "second"}]
print("page  :", await resource.list_notes(limit=2))

page  : [{'id': 'n1', 'text': 'first'}, {'id': 'n2', 'text': 'second'}]


## 5. A page with no match

No match is an empty list, not nothing, so the caller iterates it without a guard.

In [5]:
connection.rows = []
print("empty :", await resource.list_notes())

empty : []


## 6. A count

The value helper answers the first value of the first row — the number itself, not the row.

In [6]:
connection.value = 2
print("count :", await resource.count_notes())

count : 2


## 7. No value, and a closed resource

A count over nothing answers the helper's `None`, which the domain turns into `0`. After `close()` the resource has no connection, and the helper refuses with the framework's named error instead of an attribute error.

In [7]:
connection.value = None
print("absent:", await resource.count_notes())

await resource.close()
await pool.close()
try:
    await resource.list_notes()
except HandleError as error:
    print("closed:", error)

print("calls :", connection.calls)
print("borrowed:", len(pool.held))

absent: 0
closed: Connection is not open
calls : [('SELECT id, text FROM notes ORDER BY id LIMIT $1 OFFSET $2', (2, 0)), ('SELECT id, text FROM notes ORDER BY id LIMIT $1 OFFSET $2', (2, 0)), ('SELECT count(*) FROM notes', ()), ('SELECT count(*) FROM notes', ())]
borrowed: 0


## 8. Explain the result

`_fetch` and `_fetchval` are the two answers the driver did not expose before. Each takes its connection through one check, wraps a driver failure with its cause, and answers a plain shape; a method that wants its own context catches `HandleError` and raises its own. The pool owns connections, the resource owns the call, and the domain owns the meaning of absence.